In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a GPU program that "dequantizes" a weight matrix on the GPU. You are given an input matrix <code>X</code> of shape <code>[M, N]</code> containing quantized values and a scale matrix <code>S</code> of shape <code>[ceil(M/T), ceil(N/T)]</code>, where <code>T</code> is the tile size.
</p>
<p>
  For each element $X_{i,j}$, the corresponding scale factor is $S_{row, col}$ where $row = \lfloor i / T \rfloor$ and $col = \lfloor j / T \rfloor$.
  The output $Y_{i,j}$ should be computed as:
  $$
    Y_{i,j} = X_{i,j} \times S_{row, col}
  $$
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the output buffer <code>Y</code></li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:
M = 4, N = 4, TILE_SIZE = 2
X = [
  [10, 10,  5,  5],
  [10, 10,  5,  5],
  [ 2,  2,  8,  8],
  [ 2,  2,  8,  8]
]
S = [
  [0.5, 2.0],
  [4.0, 0.25]
]

Output:
Y = [
  [ 5.0,  5.0, 10.0, 10.0],
  [ 5.0,  5.0, 10.0, 10.0],
  [ 8.0,  8.0,  2.0,  2.0],
  [ 8.0,  8.0,  2.0,  2.0]
]
Explanation:
Tile (0,0) of X is multiplied by S[0,0] (0.5).
Tile (0,1) of X is multiplied by S[0,1] (2.0).
Tile (1,0) is multiplied by S[1,0] (4.0).
Tile (1,1) is multiplied by S[1,1] (0.25).
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>M</code>, <code>N</code> &le; 8192</li>
  <li><code>TILE_SIZE</code> &in; {16, 32, 64, 128}</li>

  <li>Performance is measured with <code>M</code> = 8,192, <code>N</code> = 8,192, <code>TILE_SIZE</code> = 128</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// X, S, Y are device pointers
extern "C" void solve(const float* X, const float* S, float* Y, int M, int N, int TILE_SIZE) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# X, S, Y are tensors on the GPU
@cute.jit
def solve(
    X: cute.Tensor,
    S: cute.Tensor,
    Y: cute.Tensor,
    M: cute.Int32,
    N: cute.Int32,
    TILE_SIZE: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# X, S are tensors on the GPU
@jax.jit
def solve(X: jax.Array, S: jax.Array, M: int, N: int, TILE_SIZE: int) -> jax.Array:
    # return output tensor Y directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# X, S, Y are device pointers
@export
def solve(
    X: UnsafePointer[Float32, MutExternalOrigin],
    S: UnsafePointer[Float32, MutExternalOrigin],
    Y: UnsafePointer[Float32, MutExternalOrigin],
    M: Int32,
    N: Int32,
    TILE_SIZE: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# X, S, Y are tensors on the GPU
def solve(X: torch.Tensor, S: torch.Tensor, Y: torch.Tensor, M: int, N: int, TILE_SIZE: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# X, S, Y are tensors on the GPU
def solve(X: torch.Tensor, S: torch.Tensor, Y: torch.Tensor, M: int, N: int, TILE_SIZE: int):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/medium/64_weight_dequantization/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
